In [29]:
!pip install diffusers transformers accelerate peft safetensors pillow torchvision

In [30]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

from diffusers import StableDiffusionPipeline, DDPMScheduler
from peft import LoraConfig, get_peft_model

In [31]:
# -------- CONFIG --------
MODEL_ID = "runwayml/stable-diffusion-v1-5"
# Option 1: Absolute path
DATA_DIR = "/Users/rujutabhanose/Documents/GenAI_Lab/Assignment_5/data/images"
OUTPUT_DIR = "lora_output"

IMAGE_SIZE = 256        # Reduced from 512 to save memory
BATCH_SIZE = 1          # DO NOT INCREASE
EPOCHS = 5
LR = 1e-4

# Use CPU - MPS doesn't have enough memory for SD 1.5
DEVICE = "cpu"
DTYPE = torch.float32

print(f"Using device: {DEVICE}")

Using device: cpu


In [32]:
class CustomImageDataset(Dataset):
    def __init__(self, image_dir):
        self.paths = [
            os.path.join(image_dir, f)
            for f in os.listdir(image_dir)
            if f.lower().endswith((".jpg", ".png", ".jpeg"))
        ]

        assert len(self.paths) > 0, "❌ No images found!"

        self.transform = transforms.Compose([
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5])
        ])

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        return self.transform(img)

In [33]:
dataset = CustomImageDataset(DATA_DIR)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print("Images found:", len(dataset))

Images found: 3


In [34]:
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None
)

pipe = pipe.to(DEVICE)

pipe.enable_attention_slicing("max")  # Memory savings
pipe.vae.requires_grad_(False)
pipe.text_encoder.requires_grad_(False)

# Enable gradient checkpointing for memory efficiency
pipe.unet.enable_gradient_checkpointing()

Loading weights: 100%|██████████| 196/196 [00:04<00:00, 44.05it/s, Materializing param=text_model.final_layer_norm.weight]
CLIPTextModel LOAD REPORT from: /Users/rujutabhanose/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading pipeline components...: 100%|██████████| 6/6 [00:06<00:00,  1.01s/it]
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion.StableDiffusionPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both th

In [35]:
unet = pipe.unet

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["to_q", "to_k", "to_v"],
    lora_dropout=0.1,
    bias="none"
)

unet = get_peft_model(unet, lora_config)
unet.train()

print("Trainable parameters:")
unet.print_trainable_parameters()

Trainable parameters:
trainable params: 1,195,008 || all params: 860,715,972 || trainable%: 0.1388


In [36]:
noise_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)
optimizer = torch.optim.AdamW(unet.parameters(), lr=LR)

In [37]:
# Create unconditional text embeddings (empty prompt)
with torch.no_grad():
    uncond_input = pipe.tokenizer(
        [""] * BATCH_SIZE,
        padding="max_length",
        max_length=pipe.tokenizer.model_max_length,
        return_tensors="pt"
    )
    uncond_embeddings = pipe.text_encoder(uncond_input.input_ids.to(DEVICE))[0]

for epoch in range(EPOCHS):
    for step, images in enumerate(loader):
        images = images.to(DEVICE, dtype=DTYPE)

        # VAE encode (no grad)
        with torch.no_grad():
            latents = pipe.vae.encode(images).latent_dist.sample()
            latents = latents * 0.18215

        noise = torch.randn_like(latents)
        timesteps = torch.randint(
            0,
            noise_scheduler.num_train_timesteps,
            (latents.shape[0],),
            device=DEVICE
        )

        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # Pass encoder_hidden_states to UNet
        noise_pred = unet(
            noisy_latents,
            timesteps,
            encoder_hidden_states=uncond_embeddings
        ).sample

        loss = torch.nn.functional.mse_loss(noise_pred, noise)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        print(f"Epoch {epoch+1}/{EPOCHS} | Step {step} | Loss {loss.item():.4f}")

KeyboardInterrupt: 

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
unet.save_pretrained(OUTPUT_DIR)
print("✅ LoRA weights saved to", OUTPUT_DIR)

In [ ]:
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    safety_checker=None
).to(DEVICE)

pipe.unet.load_attn_procs(OUTPUT_DIR)

prompt = "a photo in my custom dataset style"

image = pipe(prompt, num_inference_steps=30).images[0]
image